# 06 — SaulLM Investigation: fixing the "neutral-on-everything" collapse

`adrienbrault/saul-instruct-v1:Q8_0` scored **topic-macro-F1 0.136 / risk-macro-F1 0.213**
in `05_generative_llm_runs.ipynb`. Inspecting its raw output (`§7.2.2` of the
changelog) found **four** concrete failure modes, not one vague "small model is weak":

1. `risk = neutral` on **all 2,656/2,656** rows — zero variance, majority-class collapse.
2. Topic over-generation — median 6 but p75 34, **max 188** on a 42-topic taxonomy (carpet-bombing).
3. Hallucinated labels — invents `data_retention`, `data_*` ids that aren't in the taxonomy (→ 7,440 dropped).
4. Repetition looping that survives the `num_predict=768` cap.

**Root cause:** Ollama serves this model with a **ChatML** template (`<|im_start|>`),
but Saul-Instruct-v1 is a **Mistral-7B-Instruct** finetune trained on `[INST] … [/INST]`.
`<|im_start|>` are not Mistral special tokens → the model never enters instruction-following
mode. Frozen output (#1) and looping (#4) are textbook wrong-template symptoms.

This notebook tests three **cumulative** fixes, top-to-bottom, on the same small sample:

| Group | What it adds | Attacks |
|---|---|---|
| **Baseline** | Saul as shipped (ChatML) — reference on the *same* rows | — |
| **Approach A** | Correct **Mistral `[INST]` template** (rebuilt Modelfile) | #1 #2 #3 #4 (root cause) |
| **Approach B** | A **+ strict enum schema** (topics ∈ 42 ids) **+ `repeat_penalty`** | #2 #3 #4 mechanically |
| **Approach C** | B **+ two-stage** (topics-only, then risk-only on slim context) | #1 #2 (context overload) |

> **Cumulative by design:** B builds on A, C builds on B. Each group's marginal gain
> over the previous is the signal.

---
## Where results come out

Everything writes under **`generated_files/lawgic_taxonomy/evaluation_v2/generative_runs/_saul_investigation/`**:

```
_saul_investigation/
├── baseline_chatml/   raw_predictions.csv, mini_metrics.json
├── A_template/        raw_predictions.csv, mini_metrics.json, Modelfile
├── B_template_schema/         raw_predictions.csv, mini_metrics.json
├── C_template_schema_twostage/ raw_predictions.csv, mini_metrics.json
└── saul_investigation_summary.csv   ← the head-to-head table (final cell)
```

- `raw_predictions.csv` — one row/clause: `row_id, raw_json, parsed_topics, parsed_risk, parse_ok, latency_s`. **Resumable** (re-running a group skips rows already `parse_ok=True`).
- `mini_metrics.json` — per-group scores + diagnostics (risk distribution, topic-count stats, dropped labels).
- `saul_investigation_summary.csv` — baseline vs A vs B vs C, written by the last cell.

## How to run
1. **Prereqs:** `ollama serve` running; base model pulled (`ollama pull adrienbrault/saul-instruct-v1:Q8_0`). Local M1 — quit browsers (Q8_0 is 7.7 GB).
2. Run Sections 1–4 (shared setup), then each approach group in order. **`SMOKE=True` → 30 stratified rows** (default). Flip to `False` for the full 2,656-clause pass.
3. The **Summary** cell at the bottom prints and saves the comparison.

**Nothing here overwrites `05`'s published Saul row.** These are separate, clearly-labeled configs. The template fix (A/B) is a *serving-bug* fix; **C is a different *strategy*** (two-stage) and must be reported as such in the manuscript, not as a silent replacement.


## 1. Setup — corpus, split, label arrays (shared)

Identical corpus/split/label loading to `05` so the scoring is directly comparable.

In [9]:
import os, sys, json, time, subprocess
from pathlib import Path

os.environ["LAWGIC_CORPUS_VERSION"] = "v2"   # MUST precede the core import


def find_project_root(start: Path) -> Path:
    sentinel = "generated_files/lawgic_taxonomy/lawgic_multihead_wide_v2.csv"
    for c in (start, *start.parents):
        if (c / sentinel).exists():
            return c
    raise FileNotFoundError("Run this notebook from inside the lawgic repository.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import numpy as np
import pandas as pd
from dotenv import load_dotenv

import lawgic_eval_core as core

load_dotenv(PROJECT_ROOT / ".env")

core.persist_split()
corpus = core.load_corpus()
test = core.split_frames(corpus)["test"].reset_index(drop=True)
arrays = core.label_arrays(test)

TOPIC_IDS = core.TOPIC_IDS
NAME_BY_ID = core.TOPIC_NAME_BY_ID
TAXONOMY = json.loads(core.TAXONOMY_PATH.read_text(encoding="utf-8"))

assert len(TOPIC_IDS) == 42, len(TOPIC_IDS)
assert len(test) == 2656, len(test)
assert core.HARM_CLASS_NAMES == {0: "Harmful", 1: "Neutral", 2: "Fair"}
print(f"corpus={core._CORPUS_VERSION}  topics={len(TOPIC_IDS)}  test={len(test)}")

corpus=v2  topics=42  test=2656


## 2. Config — smoke switch, model tags, output dirs (shared)

**`SMOKE=True` runs `N_SMOKE=30` stratified rows.** Flip to `False` for the full 2,656.

In [10]:
# ── SMOKE control ────────────────────────────────────────────────────────────
SMOKE = False            # True: N_SMOKE stratified rows.  False: full 2,656-clause pass.
N_SMOKE = 30            # ~30 rows/approach as requested
SMOKE_SEED = 42         # fixed -> same rows across baseline/A/B/C and across reruns

# ── Decoding (same as 05; §4.4 requires greedy determinism) ──────────────────
NUM_CTX = 12288         # prevents Ollama's silent prompt truncation on local Saul
NUM_PREDICT = 768       # caps runaway generation
REPEAT_PENALTY = 1.3    # anti-loop knob, used in B and C (compatible with temp=0.0)

# ── Model tags ───────────────────────────────────────────────────────────────
SAUL_BASE = "adrienbrault/saul-instruct-v1:Q8_0"   # as-shipped (ChatML template)
SAUL_MISTRAL = "saul-instruct-v1-mistral:Q8_0"     # A: rebuilt with [INST] template (created in A.1)

# ── Output dirs ──────────────────────────────────────────────────────────────
BASE_DIR = core.EVAL_OUT_DIR / "generative_runs" / "_saul_investigation"
DIR_BASELINE = BASE_DIR / "baseline_chatml"
DIR_A = BASE_DIR / "A_template"
DIR_B = BASE_DIR / "B_template_schema"
DIR_C = BASE_DIR / "C_template_schema_twostage"
for d in (DIR_BASELINE, DIR_A, DIR_B, DIR_C):
    d.mkdir(parents=True, exist_ok=True)

# ── Risk map + pseudo-logit magnitudes (identical to 05) ─────────────────────
RISK_TO_CLASS = {"harmful": 0, "neutral": 1, "fair": 2}
DEFAULT_RISK_ON_FAILURE = "neutral"
POS_LOGIT, NEG_LOGIT = 10.0, -10.0
assert {v: k.capitalize() for k, v in RISK_TO_CLASS.items()} == core.HARM_CLASS_NAMES

print("Output base:", BASE_DIR)
print(f"SMOKE={SMOKE}  N_SMOKE={N_SMOKE}")

Output base: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation_v2/generative_runs/_saul_investigation
SMOKE=False  N_SMOKE=30


## 3. Shared helpers — prompts, schemas, parsing, run harness (shared)

Four groups of helpers reused by every approach:
- **Prompts:** `build_prompt` (single-prompt, full rubric — baseline/A) and the two-stage
  `build_topic_prompt` (slim taxonomy, no rubric) / `build_risk_prompt` (rubrics of matched topics only — C).
- **Schemas:** `ClausePrediction` (loose, free-string topics) and `ClausePredictionStrict` /
  `TopicsOnly` / `RiskOnly` (strict — topics constrained to the 42 real ids via an enum grammar — B/C).
- **Parsing / pseudo-logits:** identical to 05 (`+10/-10` trick so the scoring module is reused unchanged).
- **Run harness:** `run_approach` (resumable per-row CSV) + `score_and_report` (metrics + diagnostics).

In [11]:
import re
from enum import Enum
from typing import Literal
from pydantic import BaseModel, Field


# ── Taxonomy rendering: full (id+name+desc+rubric) vs slim (id+name+desc) ─────
def render_taxonomy_block(taxonomy, topic_ids, with_rubric=True):
    by_id = {t["id"]: t for t in taxonomy["topics"]}
    lines = []
    for tid in topic_ids:
        t = by_id[tid]
        lines.append(f"- id: {tid}")
        lines.append(f"  name: {t.get('name', tid)}")
        lines.append(f"  description: {(t.get('description') or '').strip()}")
        if with_rubric:
            lines.append("  risk_rubric:")
            for s in t.get("scores", []):
                lines.append(f"    - {s['label']}: {s['explanation'].strip()}")
    return "\n".join(lines)


TAXONOMY_BLOCK_FULL = render_taxonomy_block(TAXONOMY, TOPIC_IDS, with_rubric=True)
TAXONOMY_BLOCK_SLIM = render_taxonomy_block(TAXONOMY, TOPIC_IDS, with_rubric=False)

INSTRUCTION = (
    "You are a strict classifier of Terms-of-Service (ToS) clauses.\n"
    f"Below is a taxonomy of {len(TOPIC_IDS)} topics, each with a name, description "
    "and a 3-level risk rubric.\n"
    "Given the clause, return (1) every applicable topic ID from the taxonomy, and "
    "(2) a SINGLE overall risk class for the whole clause: exactly one of "
    "harmful, neutral, fair.\n"
    "Use only topic IDs that appear in the taxonomy. Return strict JSON only.\n"
)


def build_prompt(clause_text):
    """Single-prompt (baseline + A): full taxonomy inline, topics + risk in one call."""
    return (INSTRUCTION + "\nTAXONOMY:\n" + TAXONOMY_BLOCK_FULL
            + "\n\nCLAUSE:\n" + str(clause_text).strip()
            + '\n\nReturn JSON only: {"topics": ["<id>", ...], "risk": "harmful|neutral|fair"}')


# ── Two-stage prompts (C) ────────────────────────────────────────────────────
def build_topic_prompt(clause_text):
    """Stage 1: topics only, SLIM taxonomy (no rubric) -> smaller context for a 7B."""
    return ("You are a strict classifier of Terms-of-Service (ToS) clauses.\n"
            f"Below is a taxonomy of {len(TOPIC_IDS)} topics (id, name, description).\n"
            "Return every applicable topic ID for the clause. Use only IDs in the taxonomy.\n"
            "Return strict JSON only.\n"
            "\nTAXONOMY:\n" + TAXONOMY_BLOCK_SLIM
            + "\n\nCLAUSE:\n" + str(clause_text).strip()
            + '\n\nReturn JSON only: {"topics": ["<id>", ...]}')


def build_risk_prompt(clause_text, topic_ids_for_clause):
    """Stage 2: ONE risk class, context = only the matched topics' rubrics."""
    by_id = {t["id"]: t for t in TAXONOMY["topics"]}
    rl = []
    for tid in topic_ids_for_clause:
        t = by_id.get(tid)
        if not t:
            continue
        rl.append(f"- {tid} ({t.get('name', tid)}):")
        for s in t.get("scores", []):
            rl.append(f"    {s['label']}: {s['explanation'].strip()}")
    rubric = "\n".join(rl) if rl else "(no matched topics)"
    return ("You assess the overall risk of a Terms-of-Service (ToS) clause.\n"
            "Return ONE overall risk class for the whole clause: exactly one of "
            "harmful, neutral, fair.\n"
            "Relevant topic risk rubrics:\n" + rubric
            + "\n\nCLAUSE:\n" + str(clause_text).strip()
            + '\n\nReturn JSON only: {"risk": "harmful|neutral|fair"}')


# ── Output schemas ───────────────────────────────────────────────────────────
class ClausePrediction(BaseModel):                     # loose: baseline + A
    topics: list[str] = Field(default_factory=list)
    risk: Literal["harmful", "neutral", "fair"]


TopicEnum = Enum("TopicEnum", {tid: tid for tid in TOPIC_IDS}, type=str)   # 42 real ids


class ClausePredictionStrict(BaseModel):               # strict: B
    topics: list[TopicEnum] = Field(default_factory=list)
    risk: Literal["harmful", "neutral", "fair"]


class TopicsOnly(BaseModel):                           # C stage 1
    topics: list[TopicEnum] = Field(default_factory=list)


class RiskOnly(BaseModel):                             # C stage 2
    risk: Literal["harmful", "neutral", "fair"]


# ── Topic lookup + pseudo-logit builders (identical to 05) ───────────────────
def _norm(s):
    return re.sub(r"[\s_]+", " ", str(s).strip().lower())


def build_topic_lookup(topic_ids, name_by_id):
    lut = {}
    for idx, tid in enumerate(topic_ids):
        lut[_norm(tid)] = idx
        nm = name_by_id.get(tid)
        if nm:
            lut[_norm(nm)] = idx
    return lut


TOPIC_LOOKUP = build_topic_lookup(TOPIC_IDS, NAME_BY_ID)


def map_topics(pred_topics):
    idxs, dropped = [], []
    for t in (pred_topics or []):
        i = TOPIC_LOOKUP.get(_norm(t))
        (idxs.append(i) if i is not None else dropped.append(t))
    return sorted(set(idxs)), dropped


def topic_logits_row(idxs):
    row = np.full(len(TOPIC_IDS), NEG_LOGIT, dtype=np.float32)
    if idxs:
        row[idxs] = POS_LOGIT
    return row


def harm_logits_row(risk):
    row = np.full(3, NEG_LOGIT, dtype=np.float32)
    row[RISK_TO_CLASS[risk if risk in RISK_TO_CLASS else DEFAULT_RISK_ON_FAILURE]] = POS_LOGIT
    return row

In [13]:
from langchain_ollama import ChatOllama


def build_llm(model, schema=None, repeat_penalty=None):
    """One builder for every group. schema=None -> plain JSON mode (loose);
    schema=<pydantic> -> Ollama enforces its JSON schema (enum -> grammar)."""
    kw = dict(model=model, base_url="http://localhost:11434", temperature=0.0,
              num_ctx=NUM_CTX, num_predict=NUM_PREDICT)
    if repeat_penalty is not None:
        kw["repeat_penalty"] = repeat_penalty
    if schema is None:
        return ChatOllama(format="json", **kw)
    return ChatOllama(**kw).with_structured_output(schema)


def _topic_values(topics):
    # Enum members (strict schema) -> their .value; plain strings pass through.
    return [getattr(t, "value", t) for t in (topics or [])]


def predict_single(llm, clause_text):
    """One greedy pass (baseline + A + B). Never crashes -> parse_ok=False fallback."""
    t0 = time.perf_counter()
    topics, risk, raw, ok = [], DEFAULT_RISK_ON_FAILURE, "", True
    try:
        out = llm.invoke(build_prompt(clause_text))
        topics, risk, raw = _topic_values(out.topics), out.risk, out.model_dump_json()
    except Exception as e:
        ok, raw = False, f"PARSE_ERROR: {e}"[:2000]
    return {"parsed_topics": topics, "parsed_risk": risk, "raw_json": raw,
            "parse_ok": ok, "latency_s": round(time.perf_counter() - t0, 3)}


def predict_twostage(topic_llm, risk_llm, clause_text):
    """C: stage 1 topics (slim) -> stage 2 risk (rubrics of matched topics only)."""
    t0 = time.perf_counter()
    topics, risk, ok, raws = [], DEFAULT_RISK_ON_FAILURE, True, {}
    try:
        o1 = topic_llm.invoke(build_topic_prompt(clause_text))
        topics = _topic_values(o1.topics)
        raws["stage1"] = o1.model_dump_json()
    except Exception as e:
        ok = False
        raws["stage1"] = f"PARSE_ERROR: {e}"[:1000]
    try:
        idxs, _ = map_topics(topics)
        matched = [TOPIC_IDS[i] for i in idxs]
        o2 = risk_llm.invoke(build_risk_prompt(clause_text, matched))
        risk = o2.risk
        raws["stage2"] = o2.model_dump_json()
    except Exception as e:
        ok = False
        raws["stage2"] = f"PARSE_ERROR: {e}"[:1000]
    return {"parsed_topics": topics, "parsed_risk": risk, "raw_json": json.dumps(raws),
            "parse_ok": ok, "latency_s": round(time.perf_counter() - t0, 3)}

In [14]:
def _read_raw(raw_csv):
    df = pd.read_csv(raw_csv)
    df["row_id"] = df["row_id"].astype(int)
    df["parse_ok"] = df["parse_ok"].map(lambda x: str(x).strip().lower() in ("true", "1"))
    return df.drop_duplicates("row_id", keep="last")


def run_approach(out_dir, predict_fn, positions, verbose=True):
    """Resumable: append each clause to raw_predictions.csv; skip done row_ids."""
    raw_csv = out_dir / "raw_predictions.csv"
    done = set()
    if raw_csv.exists():
        prev = _read_raw(raw_csv)
        done = set(prev.loc[prev["parse_ok"], "row_id"])
    for n, p in enumerate(positions, 1):
        rid = int(test.iloc[p]["row_id"])
        if rid in done:
            continue
        rec = predict_fn(test.iloc[p]["text"])
        row = {"row_id": rid, "raw_json": rec["raw_json"],
               "parsed_topics": json.dumps(rec["parsed_topics"]),
               "parsed_risk": rec["parsed_risk"], "parse_ok": rec["parse_ok"],
               "latency_s": rec["latency_s"]}
        pd.DataFrame([row]).to_csv(raw_csv, mode="a", header=not raw_csv.exists(), index=False)
        done.add(rid)
        if verbose:
            print(f"  [{n}/{len(positions)}] row_id={rid} risk={rec['parsed_risk']} "
                  f"n_topics={len(rec['parsed_topics'])} ok={rec['parse_ok']} {rec['latency_s']}s")
    return raw_csv


def assemble_logits(raw, frame):
    by_id = raw.set_index("row_id")
    tl, hl, dropped, failures = [], [], 0, 0
    for rid in frame["row_id"].astype(int):
        rec = by_id.loc[rid]
        topics = rec["parsed_topics"]
        topics = json.loads(topics) if isinstance(topics, str) else list(topics or [])
        idxs, drops = map_topics(topics)
        dropped += len(drops)
        failures += int(not bool(rec["parse_ok"]))
        tl.append(topic_logits_row(idxs))
        hl.append(harm_logits_row(rec["parsed_risk"]))
    return np.vstack(tl), np.vstack(hl), dropped, failures


def score_and_report(name, out_dir, positions):
    """Score against the same rows + print diagnostics that target the 4 failure modes."""
    raw = _read_raw(out_dir / "raw_predictions.csv")
    frame = test.iloc[positions]
    raw = raw[raw["row_id"].isin(frame["row_id"])]
    tl, hl, dropped, failures = assemble_logits(raw, frame)
    m = core.all_metrics(tl, hl, {k: v[positions] for k, v in arrays.items()})
    n_topics = raw["parsed_topics"].map(lambda s: len(json.loads(s)) if isinstance(s, str) else 0)
    risk_counts = raw["parsed_risk"].value_counts().to_dict()
    rec = {
        "approach": name, "n": int(len(frame)),
        "topic_macro_f1": round(float(m["topic_macro_f1"]), 3),
        "topic_micro_f1": round(float(m["topic_micro_f1"]), 3),
        "risk_accuracy": round(float(m["risk_accuracy"]), 3),
        "risk_macro_f1": round(float(m["risk_macro_f1"]), 3),
        "risk_distribution": risk_counts,          # failure mode #1: want >1 class
        "n_risk_classes_used": len(risk_counts),
        "topics_median": float(n_topics.median()), # failure mode #2: want small
        "topics_max": int(n_topics.max()),
        "dropped_labels": int(dropped),            # failure mode #3: want 0
        "parse_failures": int(failures),
        "wall_min": round(float(raw["latency_s"].sum()) / 60, 2),
    }
    (out_dir / "mini_metrics.json").write_text(json.dumps(rec, indent=2))
    print(json.dumps(rec, indent=2))
    return rec

## 4. Sample selection — one fixed set of rows for every group (shared)

Stratified so all three observed risk classes appear (otherwise "risk variance" is meaningless). The **same** `POSITIONS` feed baseline/A/B/C, so the comparison is apples-to-apples.

In [15]:
def stratified_positions(n, seed):
    rng = np.random.default_rng(seed)
    harm, hmask = arrays["harm_labels"], arrays["harm_masks"].astype(bool)
    picks = []
    for cls in (0, 1, 2):                      # guarantee one of each observed class
        cand = np.where((harm == cls) & hmask)[0]
        if len(cand):
            picks.append(int(rng.choice(cand)))
    pool = np.where(hmask)[0].tolist()
    rng.shuffle(pool)
    for p in pool:
        if len(picks) >= n:
            break
        if p not in picks:
            picks.append(p)
    return picks[:n]


POSITIONS = stratified_positions(N_SMOKE, SMOKE_SEED) if SMOKE else list(range(len(test)))
RESULTS = {}   # approach name -> metrics rec (filled by each group; read by Summary)

gold = pd.Series([core.HARM_CLASS_NAMES.get(int(c)) for c in arrays["harm_labels"][POSITIONS]])
print(f"SMOKE={SMOKE}  n={len(POSITIONS)}")
print("gold risk distribution:", gold.value_counts().to_dict())

SMOKE=False  n=2656
gold risk distribution: {'Neutral': 1246, 'Harmful': 832, 'Fair': 578}


---
## Baseline — Saul as shipped (ChatML), same rows  ·  *reference point*

**Not one of the three fixes** — this reruns the broken `05` config on the *same* 30 rows
so the summary table compares like-for-like (the published `05` number is a 2,656-row
aggregate). Expect the four failure modes to reappear here: risk all-`neutral`, huge
`topics_max`, non-zero `dropped_labels`.

→ writes **`_saul_investigation/baseline_chatml/`**

In [7]:
# BASELINE  (SAUL_BASE = ChatML template, loose schema — exactly the 05 setup)
baseline_llm = build_llm(SAUL_BASE, schema=ClausePrediction)
run_approach(DIR_BASELINE, lambda txt: predict_single(baseline_llm, txt), POSITIONS)
RESULTS["baseline_chatml"] = score_and_report("baseline_chatml", DIR_BASELINE, POSITIONS)

  [1/30] row_id=3300 risk=neutral n_topics=7 ok=True 45.127s
  [2/30] row_id=19775 risk=neutral n_topics=114 ok=True 60.525s
  [3/30] row_id=17804 risk=neutral n_topics=1 ok=True 2.821s
  [4/30] row_id=7551 risk=neutral n_topics=2 ok=True 3.573s
  [5/30] row_id=21002 risk=neutral n_topics=1 ok=True 2.549s
  [6/30] row_id=25746 risk=neutral n_topics=3 ok=True 3.361s
  [7/30] row_id=2304 risk=neutral n_topics=3 ok=True 2.93s
  [8/30] row_id=13324 risk=neutral n_topics=38 ok=True 15.726s
  [9/30] row_id=1142 risk=neutral n_topics=1 ok=True 2.071s
  [10/30] row_id=14768 risk=neutral n_topics=29 ok=True 11.716s
  [11/30] row_id=17287 risk=neutral n_topics=2 ok=True 2.803s
  [12/30] row_id=15651 risk=neutral n_topics=4 ok=True 2.549s
  [13/30] row_id=9806 risk=neutral n_topics=1 ok=True 2.602s
  [14/30] row_id=23139 risk=neutral n_topics=111 ok=True 44.428s
  [15/30] row_id=3994 risk=neutral n_topics=1 ok=True 2.702s
  [16/30] row_id=22281 risk=neutral n_topics=16 ok=True 6.387s
  [17/30] ro

---
# ══════════════ APPROACH A — Mistral `[INST]` template ══════════════

**The root-cause fix.** Rebuild the model with the correct Mistral instruct template
instead of Ollama's bundled ChatML. **Same loose schema and same single prompt as the
baseline** — so any change here is attributable to the template alone.

- **A.1** — write a `Modelfile` and `ollama create` the `[INST]`-templated model.
- **A.2** — run the 30 rows and score.

→ writes **`_saul_investigation/A_template/`** (incl. the `Modelfile` used)

### A.1 — Rebuild Saul with the `[INST]` template

Idempotent: safe to re-run. Verifies the new template took effect.

In [16]:
# A.1 — create SAUL_MISTRAL from SAUL_BASE with Mistral's [INST] template.
MODELFILE = (
    "FROM __BASE__\n"
    'TEMPLATE "[INST] {{ .Prompt }} [/INST]"\n'
    'PARAMETER stop "</s>"\n'
    'PARAMETER stop "[INST]"\n'
).replace("__BASE__", SAUL_BASE)

mf_path = DIR_A / "Modelfile"
mf_path.write_text(MODELFILE)

res = subprocess.run(["ollama", "create", SAUL_MISTRAL, "-f", str(mf_path)],
                     capture_output=True, text=True)
print((res.stdout or res.stderr).strip()[-600:])
assert res.returncode == 0, f"ollama create failed: {res.stderr[-500:]}"

# Verify the template actually changed to [INST] (not ChatML).
shown = subprocess.run(["ollama", "show", SAUL_MISTRAL, "--template"],
                       capture_output=True, text=True).stdout
print("\n--- active template ---\n" + shown.strip())
assert "[INST]" in shown and "im_start" not in shown, "template did not switch to [INST]"
print(f"\nOK: {SAUL_MISTRAL} now serves the [INST] template.")

gathering model components 
using existing layer sha256:2e46e1dd849eeaa2fd4996b1c6e035a78b31b9510658fb42526c6745d872e6be 
using existing layer sha256:68693db5eb3e0501c644080a545730fc93d2ca2dfddf03633642b99f3a1f0e3c 
using existing layer sha256:3307d216763acfa1df163d6ddf7d764a13755b1c0040b511246b3d9fd3c00fad 
writing manifest 
success 

--- active template ---
[INST] {{ .Prompt }} [/INST]

OK: saul-instruct-v1-mistral:Q8_0 now serves the [INST] template.


### A.2 — Run + score Approach A

In [9]:
a_llm = build_llm(SAUL_MISTRAL, schema=ClausePrediction)   # [INST] template, loose schema
run_approach(DIR_A, lambda txt: predict_single(a_llm, txt), POSITIONS)
RESULTS["A_template"] = score_and_report("A_template", DIR_A, POSITIONS)

  [1/30] row_id=3300 risk=neutral n_topics=0 ok=True 30.945s
  [2/30] row_id=19775 risk=harmful n_topics=0 ok=True 1.077s
  [3/30] row_id=17804 risk=neutral n_topics=0 ok=True 0.951s
  [4/30] row_id=7551 risk=neutral n_topics=0 ok=True 1.243s
  [5/30] row_id=21002 risk=neutral n_topics=0 ok=True 1.096s
  [6/30] row_id=25746 risk=neutral n_topics=0 ok=True 1.214s
  [7/30] row_id=2304 risk=neutral n_topics=0 ok=True 1.062s
  [8/30] row_id=13324 risk=neutral n_topics=0 ok=True 1.082s
  [9/30] row_id=1142 risk=neutral n_topics=0 ok=True 0.886s
  [10/30] row_id=14768 risk=neutral n_topics=0 ok=True 0.928s
  [11/30] row_id=17287 risk=neutral n_topics=0 ok=True 1.373s
  [12/30] row_id=15651 risk=neutral n_topics=0 ok=True 0.905s
  [13/30] row_id=9806 risk=neutral n_topics=0 ok=True 1.254s
  [14/30] row_id=23139 risk=neutral n_topics=0 ok=True 1.266s
  [15/30] row_id=3994 risk=harmful n_topics=0 ok=True 1.439s
  [16/30] row_id=22281 risk=neutral n_topics=0 ok=True 0.901s
  [17/30] row_id=5361 

---
# ══════════════ APPROACH B — A + enum grammar + `repeat_penalty` ══════════════

Builds on A. Adds two mechanical guards:
- **Strict enum schema** (`ClausePredictionStrict`): `topics` items must be one of the
  **42 real ids** → Ollama compiles this to a grammar, making hallucinated `data_*`
  labels (failure #3) *impossible to emit* and bounding over-generation (failure #2).
- **`repeat_penalty=1.3`** → directly discourages the degenerate repetition loop (failure #4).

Same `[INST]` model (`SAUL_MISTRAL`) and same single prompt as A.

→ writes **`_saul_investigation/B_template_schema/`**

In [10]:
b_llm = build_llm(SAUL_MISTRAL, schema=ClausePredictionStrict, repeat_penalty=REPEAT_PENALTY)
run_approach(DIR_B, lambda txt: predict_single(b_llm, txt), POSITIONS)
RESULTS["B_template_schema"] = score_and_report("B_template_schema", DIR_B, POSITIONS)

  [1/30] row_id=3300 risk=neutral n_topics=1 ok=True 1.426s
  [2/30] row_id=19775 risk=neutral n_topics=1 ok=True 1.551s
  [3/30] row_id=17804 risk=neutral n_topics=1 ok=True 1.644s
  [4/30] row_id=7551 risk=neutral n_topics=1 ok=True 1.833s
  [5/30] row_id=21002 risk=neutral n_topics=1 ok=True 1.521s
  [6/30] row_id=25746 risk=neutral n_topics=1 ok=True 1.72s
  [7/30] row_id=2304 risk=neutral n_topics=1 ok=True 1.61s
  [8/30] row_id=13324 risk=neutral n_topics=1 ok=True 1.715s
  [9/30] row_id=1142 risk=neutral n_topics=1 ok=True 1.651s
  [10/30] row_id=14768 risk=neutral n_topics=1 ok=True 1.774s
  [11/30] row_id=17287 risk=neutral n_topics=1 ok=True 2.347s
  [12/30] row_id=15651 risk=neutral n_topics=1 ok=True 1.602s
  [13/30] row_id=9806 risk=neutral n_topics=1 ok=True 1.97s
  [14/30] row_id=23139 risk=neutral n_topics=1 ok=True 2.017s
  [15/30] row_id=3994 risk=neutral n_topics=1 ok=True 2.006s
  [16/30] row_id=22281 risk=neutral n_topics=1 ok=True 2.536s
  [17/30] row_id=5361 risk

---
# ══════════════ APPROACH C — B + two-stage (topics → risk) ══════════════

Builds on B, and additionally **shrinks the per-call context** — the hypothesis that a
6.5k-token single prompt overloads a 7B model:
- **Stage 1** (`build_topic_prompt`, `TopicsOnly`): topics only, **slim** taxonomy
  (id+name+description, no rubric) → much smaller prompt.
- **Stage 2** (`build_risk_prompt`, `RiskOnly`): the single risk class, context = **only
  the matched topics' rubrics**, not all 42. Directly targets the frozen-`neutral` (failure #1).

Both stages keep the enum grammar + `repeat_penalty`. `latency_s` sums both stages.

> **Manuscript note:** C is a **different strategy** (two-stage), not just a serving fix —
> report it as its own config, never as a drop-in replacement of the single-prompt Saul row.

→ writes **`_saul_investigation/C_template_schema_twostage/`**

In [11]:
c_topic_llm = build_llm(SAUL_MISTRAL, schema=TopicsOnly, repeat_penalty=REPEAT_PENALTY)
c_risk_llm = build_llm(SAUL_MISTRAL, schema=RiskOnly, repeat_penalty=REPEAT_PENALTY)
run_approach(DIR_C, lambda txt: predict_twostage(c_topic_llm, c_risk_llm, txt), POSITIONS)
RESULTS["C_template_schema_twostage"] = score_and_report("C_template_schema_twostage", DIR_C, POSITIONS)

  [1/30] row_id=3300 risk=neutral n_topics=1 ok=True 19.959s
  [2/30] row_id=19775 risk=harmful n_topics=2 ok=True 4.728s
  [3/30] row_id=17804 risk=neutral n_topics=1 ok=True 2.853s
  [4/30] row_id=7551 risk=neutral n_topics=1 ok=True 3.056s
  [5/30] row_id=21002 risk=neutral n_topics=1 ok=True 2.851s
  [6/30] row_id=25746 risk=neutral n_topics=1 ok=True 2.982s
  [7/30] row_id=2304 risk=neutral n_topics=2 ok=True 2.606s
  [8/30] row_id=13324 risk=harmful n_topics=2 ok=True 3.003s
  [9/30] row_id=1142 risk=harmful n_topics=1 ok=True 1.753s
  [10/30] row_id=14768 risk=neutral n_topics=0 ok=True 1.429s
  [11/30] row_id=17287 risk=neutral n_topics=1 ok=True 2.493s
  [12/30] row_id=15651 risk=neutral n_topics=2 ok=True 2.487s
  [13/30] row_id=9806 risk=harmful n_topics=1 ok=True 2.675s
  [14/30] row_id=23139 risk=neutral n_topics=1 ok=True 2.905s
  [15/30] row_id=3994 risk=harmful n_topics=1 ok=True 3.191s
  [16/30] row_id=22281 risk=neutral n_topics=1 ok=True 2.377s
  [17/30] row_id=5361 

---
# ══════════════ SUMMARY — baseline vs A vs B vs C ══════════════

Head-to-head on the **same rows**. Watch four columns against the failure modes:
`n_risk_classes_used` (#1, want >1 — the "neutral on everything" fix), `topics_max`
(#2, want small), `dropped_labels` (#3, want 0), and whether the loop is gone (reflected
in `topics_max` + `wall_min`).

→ writes **`_saul_investigation/saul_investigation_summary.csv`**

In [12]:
order = ["baseline_chatml", "A_template", "B_template_schema", "C_template_schema_twostage"]
cols = ["approach", "n", "topic_macro_f1", "topic_micro_f1", "risk_accuracy",
        "risk_macro_f1", "n_risk_classes_used", "topics_median", "topics_max",
        "dropped_labels", "wall_min"]
summary = pd.DataFrame([RESULTS[k] for k in order if k in RESULTS])[cols]
display(summary)

out_csv = BASE_DIR / "saul_investigation_summary.csv"
summary.to_csv(out_csv, index=False)
print("wrote", out_csv)
print("\nReference — published full 2,656-row run (ChatML, 05): "
      "topic_macro_f1=0.136  risk_macro_f1=0.213  dropped=7440  (risk=neutral on all rows)")

,approach,n,topic_macro_f1,topic_micro_f1,risk_accuracy,risk_macro_f1,n_risk_classes_used,topics_median,topics_max,dropped_labels,wall_min
0,baseline_chatml,30,0.102,0.123,0.500,0.222,1,4.5,114,50,5.00
1,A_template,30,0.000,0.000,0.467,0.373,3,0.0,0,0,1.05
2,B_template_schema,30,0.112,0.261,0.500,0.222,1,1.0,1,0,1.78
3,C_template_schema_twostage,30,0.116,0.282,0.500,0.332,2,1.0,2,0,1.61


wrote /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation_v2/generative_runs/_saul_investigation/saul_investigation_summary.csv

Reference — published full 2,656-row run (ChatML, 05): topic_macro_f1=0.136  risk_macro_f1=0.213  dropped=7440  (risk=neutral on all rows)


---
# ══════════════ APPROACH D — C + topic-recall nudge ══════════════

**Motivation (from the smoke):** C fixed risk + well-formedness, but stage 1 collapsed to
**1–2 topics/clause** — the enum grammar + a 7B emit the *minimal* satisfying list, so topic
**recall** (and macro-F1) stayed low even though precision was clean. D changes **only the
stage-1 prompt wording** to explicitly ask for *every* applicable topic ("most clauses match
2–6, don't return just one"). Everything else is identical to C (same `[INST]` model, enum
grammar, `repeat_penalty`, two-stage, same risk stage).

Isolates the wording effect: **D vs C** = the marginal value of the recall nudge.

→ writes **`_saul_investigation/D_twostage_recall/`** and a `saul_recall_nudge_compare.csv`

## What you need to run
1. Re-run the shared setup once if this is a fresh kernel: **§1, §2, §3, §4** (they define
   `test`, `build_llm`, `run_approach`, `score_and_report`, `POSITIONS`, `RESULTS`, `SAUL_MISTRAL`, …).
   `SAUL_MISTRAL` must already exist — if not, run **A.1** once to create it.
2. Then run the **two cells below** (D.run, then D-vs-C compare). Nothing else needs re-running;
   `RESULTS["C_template_schema_twostage"]` is reused from the C cell for the comparison, so run
   the **C cell** at least once first (or the compare cell will only show D).

### D.run — nudged stage-1 prompt, two-stage, score

In [17]:
# ── APPROACH D — C + topic-recall nudge (stage-1 wording only) ────────────────
DIR_D = BASE_DIR / "D_twostage_recall"
DIR_D.mkdir(parents=True, exist_ok=True)


def build_topic_prompt_nudge(clause_text):
    """Stage 1 + recall nudge: ask for ALL applicable topics (typically 2-6), instead of
    the single minimal one the enum grammar collapsed to in C. Slim taxonomy, as in C."""
    return ("You are a strict classifier of Terms-of-Service (ToS) clauses.\n"
            f"Below is a taxonomy of {len(TOPIC_IDS)} topics (id, name, description).\n"
            "Return EVERY applicable topic ID for the clause. Most clauses match SEVERAL "
            "topics (typically 2-6) - do NOT return just one if more apply. Include every "
            "topic the clause plausibly touches. Use only IDs in the taxonomy.\n"
            "Return strict JSON only.\n"
            "\nTAXONOMY:\n" + TAXONOMY_BLOCK_SLIM
            + "\n\nCLAUSE:\n" + str(clause_text).strip()
            + '\n\nReturn JSON only: {"topics": ["<id>", ...]}')


def predict_twostage_nudge(topic_llm, risk_llm, clause_text):
    """Same as predict_twostage but stage 1 uses the recall-nudged prompt."""
    t0 = time.perf_counter()
    topics, risk, ok, raws = [], DEFAULT_RISK_ON_FAILURE, True, {}
    try:
        o1 = topic_llm.invoke(build_topic_prompt_nudge(clause_text))
        topics = _topic_values(o1.topics)
        raws["stage1"] = o1.model_dump_json()
    except Exception as e:
        ok = False
        raws["stage1"] = f"PARSE_ERROR: {e}"[:1000]
    try:
        idxs, _ = map_topics(topics)
        matched = [TOPIC_IDS[i] for i in idxs]
        o2 = risk_llm.invoke(build_risk_prompt(clause_text, matched))
        risk = o2.risk
        raws["stage2"] = o2.model_dump_json()
    except Exception as e:
        ok = False
        raws["stage2"] = f"PARSE_ERROR: {e}"[:1000]
    return {"parsed_topics": topics, "parsed_risk": risk, "raw_json": json.dumps(raws),
            "parse_ok": ok, "latency_s": round(time.perf_counter() - t0, 3)}


d_topic_llm = build_llm(SAUL_MISTRAL, schema=TopicsOnly, repeat_penalty=REPEAT_PENALTY)
d_risk_llm = build_llm(SAUL_MISTRAL, schema=RiskOnly, repeat_penalty=REPEAT_PENALTY)
run_approach(DIR_D, lambda txt: predict_twostage_nudge(d_topic_llm, d_risk_llm, txt), POSITIONS)
RESULTS["D_twostage_recall"] = score_and_report("D_twostage_recall", DIR_D, POSITIONS)

  [1/2656] row_id=5 risk=harmful n_topics=1 ok=True 23.074s
  [2/2656] row_id=12 risk=neutral n_topics=1 ok=True 3.192s
  [3/2656] row_id=18 risk=neutral n_topics=2 ok=True 5.371s
  [4/2656] row_id=22 risk=harmful n_topics=2 ok=True 3.442s
  [5/2656] row_id=27 risk=neutral n_topics=1 ok=True 1.821s
  [6/2656] row_id=35 risk=neutral n_topics=1 ok=True 2.181s
  [7/2656] row_id=40 risk=neutral n_topics=1 ok=True 2.269s
  [8/2656] row_id=43 risk=harmful n_topics=1 ok=True 3.619s
  [9/2656] row_id=52 risk=harmful n_topics=1 ok=True 2.074s
  [10/2656] row_id=56 risk=harmful n_topics=1 ok=True 2.056s
  [11/2656] row_id=58 risk=neutral n_topics=1 ok=True 1.757s
  [12/2656] row_id=59 risk=harmful n_topics=2 ok=True 2.424s
  [13/2656] row_id=64 risk=harmful n_topics=1 ok=True 2.179s
  [14/2656] row_id=77 risk=neutral n_topics=2 ok=True 3.962s
  [15/2656] row_id=105 risk=harmful n_topics=2 ok=True 6.036s
  [16/2656] row_id=106 risk=harmful n_topics=1 ok=True 4.093s
  [17/2656] row_id=108 risk=neu

### D-vs-C — did the nudge raise topic recall without wrecking risk?

In [18]:
cols = ["approach", "n", "topic_macro_f1", "topic_micro_f1", "risk_accuracy",
        "risk_macro_f1", "n_risk_classes_used", "topics_median", "topics_max",
        "dropped_labels", "wall_min"]
order2 = ["C_template_schema_twostage", "D_twostage_recall"]
summary_d = pd.DataFrame([RESULTS[k] for k in order2 if k in RESULTS])[cols]
display(summary_d)

out_csv_d = BASE_DIR / "saul_recall_nudge_compare.csv"
summary_d.to_csv(out_csv_d, index=False)
print("wrote", out_csv_d)
print("\nWatch: topics_median/topic_micro_f1 should RISE vs C; n_risk_classes_used should "
      "stay >1 and dropped_labels stay 0 (nudge must not reintroduce carpet-bombing).")

,approach,n,topic_macro_f1,topic_micro_f1,risk_accuracy,risk_macro_f1,n_risk_classes_used,topics_median,topics_max,dropped_labels,wall_min
0,D_twostage_recall,2656,0.292,0.303,0.451,0.326,3,1.0,3,0,120.76


wrote /Users/riki/Coding Projects/Thesis/lawgic/generated_files/lawgic_taxonomy/evaluation_v2/generative_runs/_saul_investigation/saul_recall_nudge_compare.csv

Watch: topics_median/topic_micro_f1 should RISE vs C; n_risk_classes_used should stay >1 and dropped_labels stay 0 (nudge must not reintroduce carpet-bombing).
